# Logistic Regression from Scratch

**Interview-focused reference notebook** -- Binary and multinomial logistic regression, Newton's method, gradient derivation.

## Core Intuition

Logistic regression = linear model + sigmoid squashing = probability output.

$$P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T\mathbf{x}+b)}}$$

The decision boundary is where $\sigma(z) = 0.5$, i.e., $\mathbf{w}^T\mathbf{x} + b = 0$ -- a hyperplane.

**Key formulas:**
- Cross-entropy loss: $L = -\frac{1}{n}\sum_i [y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)]$
- Gradient: $\nabla L = \frac{1}{n}X^T(\sigma(X\mathbf{w}) - \mathbf{y})$ (elegantly simple!)
- Hessian: $H = \frac{1}{n}X^T S X$ where $S = \text{diag}(\hat{y}_i(1-\hat{y}_i))$

**Connection to neural networks:** The output layer of a neural network for binary classification is exactly logistic regression applied to the last hidden layer's activations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. The Sigmoid Function

In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid."""
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))

z = np.linspace(-8, 8, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z, sigmoid(z), 'b-', linewidth=2)
axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold = 0.5')
axes[0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
axes[0].set_xlabel('z')
axes[0].set_ylabel('sigma(z)')
axes[0].set_title('Sigmoid Function')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sigmoid derivative: sigma(z) * (1 - sigma(z))
sig_z = sigmoid(z)
axes[1].plot(z, sig_z * (1 - sig_z), 'g-', linewidth=2)
axes[1].set_xlabel('z')
axes[1].set_ylabel("sigma'(z)")
axes[1].set_title("Sigmoid Derivative: sigma(z)(1 - sigma(z))")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Properties: sigma(-z) = 1 - sigma(z), sigma'(z) = sigma(z)(1 - sigma(z))")
print("Max derivative = 0.25 at z=0 -- vanishing gradient problem for large |z|")

## 2. Synthetic Data

In [ ]:
from sklearn.datasets import make_classification, make_blobs

# Binary classification data
X_bin, y_bin = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                   n_informative=2, n_clusters_per_class=1,
                                   random_state=42)

# Multi-class data
X_multi, y_multi = make_blobs(n_samples=300, centers=3, n_features=2,
                               random_state=42, cluster_std=1.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter0 = axes[0].scatter(X_bin[:, 0], X_bin[:, 1], c=y_bin, cmap='RdBu', alpha=0.7, edgecolors='k', s=40)
axes[0].set_title('Binary Classification Data')
axes[0].legend(*scatter0.legend_elements(), title='Class')

scatter1 = axes[1].scatter(X_multi[:, 0], X_multi[:, 1], c=y_multi, cmap='viridis', alpha=0.7, edgecolors='k', s=40)
axes[1].set_title('Multi-class Data')
axes[1].legend(*scatter1.legend_elements(), title='Class')

plt.tight_layout()
plt.show()

## 3. Binary Logistic Regression -- Gradient Descent

### Gradient Derivation

The cross-entropy loss for sample $i$: $\ell_i = -[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)]$

where $\hat{y}_i = \sigma(\mathbf{w}^T\mathbf{x}_i)$.

Taking the derivative w.r.t. $\mathbf{w}$:

$$\frac{\partial \ell_i}{\partial \mathbf{w}} = -(y_i - \hat{y}_i)\mathbf{x}_i$$

Over the full dataset:

$$\nabla L = \frac{1}{n} X^T(\hat{\mathbf{y}} - \mathbf{y}) = \frac{1}{n} X^T(\sigma(X\mathbf{w}) - \mathbf{y})$$

This is identical in form to the linear regression gradient -- the sigmoid just "wraps" the prediction.

In [ ]:
class LogisticRegressionGD:
    """Binary logistic regression via gradient descent."""
    
    def __init__(self, lr=0.1, n_iters=1000, reg=0.0):
        self.lr = lr
        self.n_iters = n_iters
        self.reg = reg  # L2 regularization
        self.loss_history = []
    
    def _sigmoid(self, z):
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))
    
    def _cross_entropy(self, y, y_hat):
        eps = 1e-12
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    
    def fit(self, X, y):
        n, d = X.shape
        X_b = np.column_stack([np.ones(n), X])
        self.w_ = np.zeros(d + 1)
        
        for i in range(self.n_iters):
            y_hat = self._sigmoid(X_b @ self.w_)
            
            # Gradient: (1/n) X^T (sigma(Xw) - y) + regularization
            grad = (1.0 / n) * X_b.T @ (y_hat - y)
            grad[1:] += self.reg * self.w_[1:]  # don't regularize bias
            
            self.w_ -= self.lr * grad
            self.loss_history.append(self._cross_entropy(y, y_hat))
        
        self.bias_ = self.w_[0]
        self.coef_ = self.w_[1:]
        return self
    
    def predict_proba(self, X):
        X_b = np.column_stack([np.ones(X.shape[0]), X])
        return self._sigmoid(X_b @ self.w_)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

model_gd = LogisticRegressionGD(lr=0.5, n_iters=500).fit(X_bin, y_bin)
acc = np.mean(model_gd.predict(X_bin) == y_bin)
print(f"Training accuracy: {acc:.4f}")
print(f"Weights: {model_gd.coef_.round(4)}, Bias: {model_gd.bias_:.4f}")

## 4. Newton's Method / IRLS

Newton's method uses second-order information (Hessian) for faster convergence:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - H^{-1} \nabla L$$

where $H = \frac{1}{n} X^T S X$ and $S = \text{diag}(\hat{y}_i(1-\hat{y}_i))$.

Converges quadratically (vs. linearly for GD), but each step is $O(d^3)$ for the Hessian inverse.

In [ ]:
class LogisticRegressionNewton:
    """Binary logistic regression via Newton's method (IRLS)."""
    
    def __init__(self, n_iters=20):
        self.n_iters = n_iters
        self.loss_history = []
    
    def _sigmoid(self, z):
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))
    
    def _cross_entropy(self, y, y_hat):
        eps = 1e-12
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    
    def fit(self, X, y):
        n, d = X.shape
        X_b = np.column_stack([np.ones(n), X])
        self.w_ = np.zeros(d + 1)
        
        for i in range(self.n_iters):
            y_hat = self._sigmoid(X_b @ self.w_)
            self.loss_history.append(self._cross_entropy(y, y_hat))
            
            # Gradient
            grad = (1.0 / n) * X_b.T @ (y_hat - y)
            
            # Hessian: (1/n) X^T S X where S = diag(y_hat * (1 - y_hat))
            S = y_hat * (1 - y_hat)
            H = (1.0 / n) * (X_b.T * S) @ X_b  # efficient: X^T diag(S) X
            
            # Newton update: w -= H^{-1} grad
            try:
                delta = np.linalg.solve(H + 1e-8 * np.eye(d + 1), grad)
                self.w_ -= delta
            except np.linalg.LinAlgError:
                break
        
        self.bias_ = self.w_[0]
        self.coef_ = self.w_[1:]
        return self
    
    def predict_proba(self, X):
        X_b = np.column_stack([np.ones(X.shape[0]), X])
        return self._sigmoid(X_b @ self.w_)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

model_newton = LogisticRegressionNewton(n_iters=20).fit(X_bin, y_bin)
acc = np.mean(model_newton.predict(X_bin) == y_bin)
print(f"Newton's method accuracy: {acc:.4f}")

# Compare convergence speed
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model_gd.loss_history[:100], label='Gradient Descent', linewidth=2)
ax.plot(model_newton.loss_history, 'r-o', label="Newton's Method", linewidth=2, markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Convergence: GD vs Newton (note: Newton converges in ~10 steps)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(model, X, y, ax, title=''):
    """Plot decision boundary for a 2D binary classifier."""
    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=40, alpha=0.7)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(model_gd, X_bin, y_bin, axes[0], 'Gradient Descent')
plot_decision_boundary(model_newton, X_bin, y_bin, axes[1], "Newton's Method")
plt.tight_layout()
plt.show()
print("Both give the same linear decision boundary -- the boundary is where w^T x + b = 0.")

## 6. Multinomial Logistic Regression

For $K$ classes, we maintain a weight matrix $W \in \mathbb{R}^{d \times K}$:

$$P(y=k|\mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^K e^{\mathbf{w}_j^T \mathbf{x}}} = \text{softmax}(\mathbf{z})_k$$

Loss: cross-entropy over all classes

$$L = -\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K} y_{ik} \log \hat{y}_{ik}$$

In [ ]:
class MultinomialLogisticRegression:
    """Multinomial (multi-class) logistic regression via gradient descent."""
    
    def __init__(self, lr=0.1, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.loss_history = []
    
    def _softmax(self, Z):
        """Numerically stable softmax."""
        Z_shifted = Z - Z.max(axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        return exp_Z / exp_Z.sum(axis=1, keepdims=True)
    
    def _one_hot(self, y, K):
        n = y.shape[0]
        Y = np.zeros((n, K))
        Y[np.arange(n), y] = 1
        return Y
    
    def fit(self, X, y):
        n, d = X.shape
        self.K_ = len(np.unique(y))
        X_b = np.column_stack([np.ones(n), X])
        Y = self._one_hot(y, self.K_)
        
        # Weight matrix: (d+1) x K
        self.W_ = np.zeros((d + 1, self.K_))
        
        for i in range(self.n_iters):
            # Forward
            probs = self._softmax(X_b @ self.W_)  # n x K
            
            # Loss
            eps = 1e-12
            loss = -np.mean(np.sum(Y * np.log(probs + eps), axis=1))
            self.loss_history.append(loss)
            
            # Gradient: (1/n) X^T (P - Y)
            grad = (1.0 / n) * X_b.T @ (probs - Y)
            self.W_ -= self.lr * grad
        
        return self
    
    def predict_proba(self, X):
        X_b = np.column_stack([np.ones(X.shape[0]), X])
        return self._softmax(X_b @ self.W_)
    
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

model_multi = MultinomialLogisticRegression(lr=0.1, n_iters=1000).fit(X_multi, y_multi)
acc = np.mean(model_multi.predict(X_multi) == y_multi)
print(f"Multi-class accuracy: {acc:.4f}")

# Visualize decision regions
fig, ax = plt.subplots(figsize=(8, 6))
h = 0.1
x_min, x_max = X_multi[:, 0].min() - 2, X_multi[:, 0].max() + 2
y_min, y_max = X_multi[:, 1].min() - 2, X_multi[:, 1].max() + 2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = model_multi.predict(np.column_stack([xx.ravel(), yy.ravel()]))
Z = Z.reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = ax.scatter(X_multi[:, 0], X_multi[:, 1], c=y_multi, cmap='viridis',
                     edgecolors='k', s=40, alpha=0.7)
ax.set_title('Multinomial Logistic Regression Decision Regions')
ax.legend(*scatter.legend_elements(), title='Class')
plt.tight_layout()
plt.show()

## 7. Loss Surface Visualization

In [ ]:
# 1D slice of loss surface for binary logistic regression
X_simple = np.array([[1], [2], [3], [4], [5], [6]], dtype=float)
y_simple = np.array([0, 0, 0, 1, 1, 1])

w_range = np.linspace(-3, 5, 100)
b_range = np.linspace(-10, 5, 100)
W, B = np.meshgrid(w_range, b_range)
Loss = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        z = X_simple.ravel() * W[i,j] + B[i,j]
        y_hat = sigmoid(z)
        y_hat = np.clip(y_hat, 1e-12, 1 - 1e-12)
        Loss[i,j] = -np.mean(y_simple * np.log(y_hat) + (1-y_simple) * np.log(1-y_hat))

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contour(W, B, Loss, levels=30, cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8)
ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_title('Cross-Entropy Loss Surface (Convex!)')
ax.grid(True, alpha=0.3)
plt.colorbar(cs, ax=ax, label='Loss')
plt.tight_layout()
plt.show()
print("The loss surface is convex -- no local minima, guaranteed global optimum.")

## 8. Sklearn Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogistic
from sklearn.metrics import accuracy_score, classification_report

# Binary comparison
sk_bin = SkLogistic(max_iter=1000).fit(X_bin, y_bin)
print("Binary Classification:")
print(f"  Our accuracy:     {np.mean(model_gd.predict(X_bin) == y_bin):.4f}")
print(f"  sklearn accuracy: {sk_bin.score(X_bin, y_bin):.4f}")
print(f"  Our weights:     {model_gd.coef_.round(4)}")
print(f"  sklearn weights: {sk_bin.coef_.round(4)}")

print("\nMulti-class Classification:")
sk_multi = SkLogistic(max_iter=1000, multi_class='multinomial').fit(X_multi, y_multi)
print(f"  Our accuracy:     {np.mean(model_multi.predict(X_multi) == y_multi):.4f}")
print(f"  sklearn accuracy: {sk_multi.score(X_multi, y_multi):.4f}")

## 9. Interview: Why Cross-Entropy, Not MSE?

In [ ]:
# Demonstrate why MSE is problematic for classification
y_hat = np.linspace(0.001, 0.999, 200)

# For y=1:
mse_loss = (1 - y_hat) ** 2
ce_loss = -np.log(y_hat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(y_hat, mse_loss, 'b-', linewidth=2, label='MSE')
axes[0].plot(y_hat, ce_loss, 'r-', linewidth=2, label='Cross-Entropy')
axes[0].set_xlabel('Predicted probability (y_hat)')
axes[0].set_ylabel('Loss (when true label = 1)')
axes[0].set_title('Loss Comparison for y=1')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 5)

# MSE gradient with sigmoid
z = np.linspace(-6, 6, 200)
sig = sigmoid(z)
# d/dz MSE for y=1: 2(sig(z) - 1) * sig(z) * (1-sig(z))
mse_grad = 2 * (sig - 1) * sig * (1 - sig)
# d/dz CE for y=1: sig(z) - 1
ce_grad = sig - 1

axes[1].plot(z, np.abs(mse_grad), 'b-', linewidth=2, label='|MSE gradient|')
axes[1].plot(z, np.abs(ce_grad), 'r-', linewidth=2, label='|CE gradient|')
axes[1].set_xlabel('z = w^T x')
axes[1].set_ylabel('|Gradient magnitude|')
axes[1].set_title('Gradient Magnitude when y=1')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Key insight: MSE gradient vanishes when prediction is very wrong (z << 0).")
print("Cross-entropy gradient stays strong -- learning doesn't stall for confident wrong predictions.")
print("Also: MSE + sigmoid gives non-convex loss; CE + sigmoid is convex.")

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Why cross-entropy not MSE? | MSE gradient vanishes for confident wrong predictions (sigmoid saturation). CE keeps gradient strong. Also CE+sigmoid is convex; MSE+sigmoid is not. |
| Is logistic regression convex? | Yes. Cross-entropy loss with sigmoid is convex w.r.t. weights. Proof: the Hessian $X^T S X$ is positive semidefinite since $S$ has non-negative diagonal. |
| Logistic vs Naive Bayes? | LR is discriminative (models $P(y|x)$ directly). NB is generative (models $P(x|y)P(y)$). LR usually better with lots of data; NB can be better with small data or when independence holds. |
| What is the decision boundary? | A hyperplane: $\mathbf{w}^T\mathbf{x} + b = 0$. Same as SVM but different objective. |
| Connection to neural nets? | LogReg = neural net with 0 hidden layers. The output layer of a classifier is exactly logistic/softmax regression on the last hidden layer. |
| Regularization? | L2 is standard (C parameter in sklearn = 1/lambda). Prevents overfitting, especially with many features. |
| Newton vs GD? | Newton converges quadratically but costs $O(d^3)$ per step. GD costs $O(nd)$ per step. Newton preferred when $d$ is small. |